# 13. Lecke - Ügynök Memória


## Beállítás

Ez a jegyzetfüzet azt mutatja be, hogyan lehet utazási foglalási ügynököt építeni **tartós memóriával** a **Microsoft Agent Framework** (MAF) használatával.

Megtanulhatja, hogyan befolyásolják az ügynök memóriájának különböző típusai — munkamemória, rövid távú és hosszú távú memória — azt, hogy az ügynök miként őrzi meg és használja az információkat a beszélgetések során.

**Előfeltételek:**
- Egy Azure AI Foundry projekt, amelyben egy chat modell telepítve van (pl. `gpt-4o-mini`).
- Azure CLI-ben bejelentkezve — futtassa a `az login` parancsot a termináljában.
- `AZURE_AI_PROJECT_ENDPOINT` — az Azure AI Foundry projekt végpontja.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — a telepített modell neve.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Ügynök memória típusai

Az AI ügynökök különböző típusú memóriákat használhatnak, amelyek mindegyike más-más célt szolgál:

### Munkamemória
Magának a beszélgetési szálnak az üzenetei — amelyek egyetlen munkamenet során váltanak üzenetet. Az ügynök visszautalhat korábbi üzenetekre ugyanabban a szálban a koherencia fenntartása érdekében. A MAF-ban ez a **`agent.create_session()`** segítségével jön létre, amely egy `AgentSession`-t ad vissza.

### Rövid távú memória
Olyan információk, amelyek a feladat vagy munkamenet idejére megmaradnak, de nem tárolódnak tartósan. Például az ügynök összegyűjtheti a tényeket egy többlépcsős tervezési beszélgetés során, és ezeket felhasználhatja egy végleges útiterv elkészítéséhez.

### Hosszú távú memória
Olyan preferenciák és tények, amelyek **munkamenetek között** is megőrződnek. Egy visszatérő felhasználónak nem kell megismételnie az étrendi korlátozásait vagy az utazási stílusát. A hosszú távú memória jellemzően egy külső adattároló, például adatbázis, fájl vagy vektoralapú index által van támogatva, és eszközökön keresztül jelenik meg az ügynök számára.


## Munkamemória munkamenetekkel

A legegyszerűbb memóriaforma a beszélgetési munkamenet. Ha ugyanazt a munkamenetet (az `agent.create_session()` segítségével létrehozottat) adja át egymást követő `agent.run()` hívásoknak, az ügynök látja a beszélgetés teljes előzményeit, és fel tud idézni korábbi részleteket.

Hozzunk létre egy utazási ügynököt, és mutassuk be a munkamemóriát.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Az ügynök helyesen idézte fel a költségvetést, mert mindkét üzenet ugyanahhoz a munkamenethez tartozik. Ez az **aktív memória** — amely csak a munkamenet ideje alatt létezik.

### Mi történik egy új szál esetén?

Ha **új** munkamenetet hozunk létre, az ügynöknek nincs emléke az előző beszélgetésről:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Hosszú távú memória mintázat

A felhasználói beállítások **munkamenetek közötti** megjegyzéséhez olyan tartós tárolóra van szükség, amely a beszélgetési szálon kívül létezik. Az ügynök ehhez a tárolóhoz **eszközökön** keresztül fér hozzá — olyan függvényeken, amelyeket hívhat az információk mentésére és lekérésére.

Lent egy egyszerű memóriabeli preferenciatárolót valósítunk meg (éles környezetben ezt adatbázissal vagy vektorindexszel támasztanánk alá), és eszközként tesszük elérhetővé az ügynök számára.

### Architektúra
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Forgatókönyv 1 — Első alkalommal felhasználó foglal évfordulós utazást

Sarah először látogat el. Az ügynöknek el kell mentenie az általa adott preferenciákat a eszközökön keresztül, és ezeket használva kell szállodákat ajánlania.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 2. forgatókönyv — Sarah hetek múlva visszatér

Sarah **egy teljesen új szálat** indít (egy új munkamenet szimulálása). A munkamemória üres, de a hosszú távú preferenciák tárolója továbbra is tartalmazza az ő adatait. Az ügynöknek elő kell keresnie azokat, és fel kell használnia a személyre szabott ajánlásokhoz.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Összefoglaló

Ebben a leckében három típusú ügynök memóriát tanultál, és azt, hogyan valósíthatók meg a Microsoft Agent Framework segítségével:

| Memóriatípus | MAF mechanizmus | Élettartam |
|---|---|---|
| **Működő** | `agent.create_session()` | Egyetlen beszélgetés |
| **Rövid távú** | Felhalmozott kontextus egy szálon belül | Egyetlen feladat / munkamenet |
| **Hosszú távú** | Külső tároló, amelyhez `@tool` függvényekkel férünk hozzá | Több munkameneten át |

### Főbb tanulságok
1. **`agent.create_session()`** működő memóriát biztosít — az ügynök látja a teljes beszélgetési előzményt egy munkameneten belül.
2. **Új munkamenetek elvesztik a kontextust** — hosszú távú memória nélkül az ügynök nem tudja felidézni a korábbi beszélgetéseket.
3. **A `@tool` függvények hidat képeznek** — lehetővé teszik az információk mentését és előhívását egy perzisztens tárolóból.
4. **A személyre szabás idővel javul** — minél több preferencia tárolódik, annál jobb az ügynök ajánlása.

### Valós alkalmazások
- **Ügyfélszolgálat**: Az ügyfél előzményeinek és preferenciáinak megjegyzése
- **Személyi asszisztensek**: Kontextus megőrzése napokon vagy heteken át
- **Egészségügy**: Betegadatok és preferenciák nyomon követése
- **E-kereskedelem**: Személyre szabott vásárlás az előzmények alapján

### Következő lépések
- Helyettesítsd az in-memory dictionary-t adatbázissal vagy vektortárral (pl. Azure AI Search)
- Adj hozzá memóriakorlátozást időérzékeny információkhoz
- Építs többügynökös rendszereket megosztott memóriával
- Fedezd fel a Cognee jegyzetfüzetet tudásgráf alapú memóriáért


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Felelősségkizárás**:
Jelen dokumentumot az AI fordító szolgáltatás [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk le. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások tartalmazhatnak hibákat vagy pontatlanságokat. Az eredeti, anyanyelvi dokumentum tekintendő hiteles forrásnak. Kritikus információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
